<a href="https://colab.research.google.com/github/tsubasa-iino/psi4book/blob/main/compchem_book_ch07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 7章　振動数計算をしてみよう

### 環境構築

#### Google Colab上にPsi4をインストール

In [ ]:
!pip install -q condacolab
import condacolab
import os

# バグ回避パッチ
if "LD_LIBRARY_PATH" not in os.environ:
    os.environ["LD_LIBRARY_PATH"] = ""

print("Installing CondaColab (Base)...")
condacolab.install()

Installing CondaColab (Base)...
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


ランタイム｜セッションを再起動する｜はい を実行。

In [ ]:
import condacolab
import os
import sys

condacolab.check()

# 1. 邪魔なPinningを削除
if os.path.exists("/usr/local/conda-meta/pinned"):
    !rm /usr/local/conda-meta/pinned

# 2. Python 3.12 と Psi4 をインストール（ディスク書き換え）
print("Upgrading Python to 3.12 & Installing Psi4...")
!mamba install -y -q python=3.12 psi4 -c conda-forge/label/libint_dev -c conda-forge

# 3. Pinningの復元（成功環境の再現）
os.makedirs("/usr/local/conda-meta", exist_ok=True)
with open("/usr/local/conda-meta/pinned", "w") as f:
    f.write("python 3.12.*\n")

# 4. 【最重要】カーネルの自殺（強制再起動）
# これにより、メモリ上のPython 3.11を殺し、ディスク上のPython 3.12をロードさせます
print("\n🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...")
import time
time.sleep(1)
os.kill(os.getpid(), 9)

✨🍰✨ Everything looks OK!
Upgrading Python to 3.12 & Installing Psi4...
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done

🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...


ランタイム｜セッションを再起動する｜はい を実行。

In [ ]:
import sys
import os

# パスが通っていなければ通す
target_path = "/usr/local/lib/python3.12/site-packages"
if target_path not in sys.path:
    sys.path.insert(0, target_path)

import psi4
print(f"✅ Restart Successful.")
print(f"Psi4 Version: {psi4.__version__}")
print(f"Python Version: {sys.version.split()[0]}") # ここが3.12になっているはず

# 計算テスト
psi4.set_memory('500 MB')
mol = psi4.geometry("O\nH 1 0.96\nH 1 0.96 2 104.5")
en = psi4.energy('scf/cc-pvdz')
print(f"Energy: {en:.6f}")

✅ Restart Successful.
Psi4 Version: 1.10
Python Version: 3.12.12
Energy: -76.026633


ここまででインストール確認完了。

In [ ]:
import os
import datetime
import numpy as np
import pandas as pd
import psi4

print(f'current time: {datetime.datetime.now()}')
print(f'python version:\n{sys.version}')
print(f'numpy version: {np.__version__}')
print(f'pandas version: {pd.__version__}')
print(f'psi4 version: {psi4.__version__}')

current time: 2026-01-18 14:21:32.033912
python version:
3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy version: 2.0.2
pandas version: 2.2.2
psi4 version: 1.10


#### 必要なライブラリと関数の定義

In [ ]:
# 振動数可視化のためのライブラリ
!pip install py3Dmol
import py3Dmol

!git clone https://github.com/duerrsimon/normal-mode-jupyter.git
sys.path.append('/content/normal-mode-jupyter')
from helpers import show_normal_modes

Cloning into 'normal-mode-jupyter'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 23 (delta 8), reused 17 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 801.96 KiB | 7.64 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [ ]:
def show_3D(mol: psi4.core.Molecule) -> py3Dmol.view:
    """
    Psi4のMoleculeオブジェクトをpy3Dmolで描画する
    Args:
        mol: 描画対象の分子

    Returns:
        py3Dmol.view: py3Dmolの描画オブジェクト

    """
    view = py3Dmol.view(width=400, height=400)
    xyz = mol.save_string_xyz_file()
    view.addModel(xyz, 'xyz')
    view.setStyle({'stick': {}})
    view.setBackgroundColor('#e1e1e1')
    view.zoomTo()

    return view.show()

#### 計算資源の設定

In [ ]:
# 計算資源の確認（CPU, RAM）
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2200.212
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi its
bogomips	: 4400.42
clflush size	: 64
cache_alignment	: 64
address sizes

In [ ]:
!cat /proc/meminfo

MemTotal:       13286964 kB
MemFree:          632192 kB
MemAvailable:   11851728 kB
Buffers:          180420 kB
Cached:         10862932 kB
SwapCached:            0 kB
Active:          1176524 kB
Inactive:       10789064 kB
Active(anon):       2628 kB
Inactive(anon):   922960 kB
Active(file):    1173896 kB
Inactive(file):  9866104 kB
Unevictable:           8 kB
Mlocked:               8 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:             11376 kB
Writeback:             0 kB
AnonPages:        922304 kB
Mapped:           546932 kB
Shmem:              3344 kB
KReclaimable:     505180 kB
Slab:             578000 kB
SReclaimable:     505180 kB
SUnreclaim:        72820 kB
KernelStack:        5872 kB
PageTables:        16848 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:     6643480 kB
Committed_AS:    3077440 kB
VmallocTotal:   34359738367 kB
VmallocUsed:       12344 kB
VmallocChunk:    

In [ ]:
n_cpu = os.cpu_count()
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3)

In [ ]:
# 環境に応じて計算資源を設定
psi4.set_num_threads(n_cpu)
psi4.set_memory(f'{ram * 0.9: .0f}GB')

11000000000

### 7.1 振動数計算とは何だろう

#### 水分子の計算

In [ ]:
# ログファイルの設定
psi4.set_output_file('h2o_optfreq_hf-sto3g.log')

PosixPath('h2o_optfreq_hf-sto3g.log')

In [ ]:
# 水分子の構造の設定
h2o = psi4.geometry('''
0 1
O       -0.7520847362      0.3573098626      0.0156794168
H        0.2372416200      0.3907947487     -0.0110698848
H       -1.0414501312      1.0972341870     -0.5754069255
''')

# 構造最適化計算の実行
psi4.optimize('hf/sto-3g', molecule=h2o)

Optimizer: Optimization complete!


-74.96599011057671

In [ ]:
# 最適化構造の出力
print(h2o.save_string_xyz())

0 1
 O   -0.000000000000    0.000000000000   -0.071153222159
 H    0.758023512375    0.000000000000    0.564626635051
 H   -0.758023512375   -0.000000000000    0.564626635051



In [ ]:
show_3D(h2o)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
_, wfn = psi4.frequency('hf/sto-3g', molecule=h2o, return_wfn=True)
print(wfn.frequencies().to_array())

[ 2170.20249414  4140.61402693  4391.65087082]


##### normal_mode_writeを設定

In [ ]:
psi4.set_options({'normal_modes_write': True})
psi4.freq('hf/sto-3g', molecule=h2o)

-74.96599011057671

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('h2o_optfreq_hf-sto3g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((2170.2024930174, 0), (4140.6140257721, 1)…

#### エタン・エチレン・アセチレンの比較

##### エタン

In [ ]:
psi4.set_output_file('ethane_optfreq_hf-3-21g.log')

# エタン構造の定義
ethane = psi4.geometry('''
0 1
 C                 -1.25726143    1.25726139    0.00000000
 H                 -0.90060700    0.24845139    0.00000000
 H                 -0.90058859    1.76165958   -0.87365150
 H                 -2.32726143    1.25727458    0.00000000
 C                 -0.74391921    1.98321767    1.25740497
 H                  0.32608077    1.98303509    1.25750243
 H                 -1.10041427    2.99208399    1.25730739
 H                 -1.10075147    1.47893186    2.13105625
 ''')

# 構造最適化
_, wfn_c2h6 = psi4.optimize('hf/3-21g', molecule=ethane, return_wfn=True)

Optimizer: Optimization complete!


In [ ]:
print(ethane.save_string_xyz())

0 1
 C   -0.257110322933   -0.363599326471   -0.629778216862
 H    0.092333089034   -1.389657823686   -0.651309520881
 H    0.092516921875    0.130660334787   -1.529132697523
 H   -1.340968988540   -0.375958604221   -0.651410489200
 C    0.257110320104    0.363599321023    0.629778206534
 H    1.340969065566    0.375963842341    0.651407522469
 H   -0.092338015943    1.389656165246    0.651312548365
 H   -0.092512038310   -0.130663849604    1.529132759741



In [ ]:
show_3D(ethane)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数計算
psi4.set_options({'normal_modes_write': True})
_, wfn_c2h6 = psi4.frequency('hf/3-21g',
                             molecule=ethane,
                             return_wfn=True,
                             ref_gradient=wfn_c2h6.gradient())

In [ ]:
print(wfn_c2h6.frequencies().to_array())

[  313.69658563   921.68387993   921.68392341  1003.81311691
  1351.55830578  1351.55831265  1571.30785672  1579.66742578
  1677.07244931  1677.07249632  1678.02476145  1678.02482481
  3195.91660272  3200.12294422  3240.69472852  3240.69483619
  3267.53944342  3267.53955797]


In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('ethane_optfreq_hf-3-21g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((313.6965856345, 0), (921.6838799291, 1), …

##### エチレン

In [ ]:
psi4.set_output_file('ethylene_optfreq_hf-3-21g.log')

# エチレンの構造定義
ethylene = psi4.geometry('''
0 1
 C                 -0.31950208    0.90041492    0.00000000
 C                  1.00641392    0.90041492    0.00000000
 H                 -0.91308708    1.82445292    0.00000000
 H                 -0.91311808   -0.02359908   -0.00002200
 H                  1.59999892   -0.02362308   -0.00001900
 H                  1.60002992    1.82442892    0.00002600
''')

# 構造最適化
_, wfn_c2h4 = psi4.optimize('hf/3-21g',
              molecule=ethylene,
              return_wfn=True)

Optimizer: Optimization complete!


In [ ]:
show_3D(ethylene)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数計算
psi4.set_options({'normal_modes_write': True})
_, wfn_c2h4 = psi4.frequency('hf/3-21g',
                             molecule=ethylene,
                             return_wfn=True,
                             ref_gradient=wfn_c2h4.gradient())

# 振動数を出力
print(wfn_c2h4.frequencies().to_array())

[  943.48733816  1114.67812771  1156.51303721  1165.85042143
  1387.43212621  1522.31395426  1639.76156202  1842.57331573
  3305.44446479  3327.23471273  3371.16427469  3402.67129030]


In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('ethylene_optfreq_hf-3-21g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((943.4873381639, 0), (1114.6781277096, 1),…

##### アセチレン

In [ ]:
psi4.set_output_file('acetylene_optfreq_hf-3-21g.log')

# アセチレンの構造定義
acetylene = psi4.geometry('''
0 1
 C                 -0.51037345    0.81742737    0.00000000
 C                  0.68462655    0.81742737    0.00000000
 H                 -1.57137345    0.81742737    0.00000000
 H                  1.74562655    0.81742737    0.00000000
''')

# 構造最適化
_, wfn_c2h2 = psi4.optimize('hf/3-21g',
                            molecule=acetylene,
                            return_wfn=True)

Optimizer: Optimization complete!


In [ ]:
show_3D(acetylene)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数計算
psi4.set_options({'normal_modes_write': True})
_, wfn_c2h2 = psi4.frequency('hf/3-21g',
                             molecule=acetylene,
                             ref_gradient=wfn_c2h2.gradient(),
                             return_wfn=True)

In [ ]:
# 振動数を出力
print(wfn_c2h2.frequencies().to_array())

[  902.40193051   902.40193222   918.00090015   918.00091270
  2233.80197621  3596.13605495  3719.33772776]


In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('acetylene_optfreq_hf-3-21g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((902.4019305149, 0), (902.4019322227, 1), …

### 分子のIRスペクトルを求めてみよう

In [ ]:
def optfreq(mol: psi4.core.Molecule,
            theory: str) -> list[float, psi4.core.Wavefunction]:
    """
    分子の構造最適化と振動数計算を行う

    Args:
        mol (psi4.core.Molecule): 計算対象の分子
        theory: 計算レベル

    Returns:
        float: エネルギー
        psi4.core.Wavefunction: 計算のWavefunction

    """
    _, wfn = psi4.optimize(theory,
                           molecule=mol,
                           return_wfn=True)
    energy, wfn = psi4.frequency(theory,
                                 molecule=mol,
                                 ref_gradient=wfn.gradient(),
                                 return_wfn=True)

    return [energy, wfn]


def get_freqs(wavefunction: psi4.core.Wavefunction) -> np.ndarray:
    """
    Wavefunctionオブジェクトから振動数を取り出す
    Args:
        wavefunction: 対象のWavefunctionオブジェクト

    Returns:
        振動数のarrray

    """
    return wavefunction.frequencies().to_array()


# 計算レベルとログファイル名のリスト
theories = ['hf/sto-3g', 'hf/3-21g', 'mp2/6-31g', 'wb97x-d/aug-cc-pvdz']
logfiles = ['hf_sto3g', 'hf_3-21g', 'mp2_6-31g', 'wb97xd_aug-cc-pvdz']

#### アセトアルデヒド

In [ ]:
## アセトアルデヒドの構造を定義
acetaldehyde = psi4.geometry('''
0 1
 C                  0.11203320    0.61825725    0.00000000
 O                  1.33489796    0.61545082   -0.10440743
 H                 -0.48011015   -0.32114570    0.00198167
 C                 -0.70916294    1.92103771    0.00000000
 H                 -0.40304404    2.53560774   -0.82066734
 H                 -1.74874842    1.68735489   -0.09774644
 H                 -0.54626743    2.44532874    0.91841383
''')

##### HF/STO-3G

In [ ]:
psi4.set_output_file('acetaldehyde_hf-sto3g.log')
psi4.set_options({'normal_modes_write': True})

In [ ]:
# 構造最適化・振動数計算の実行
_, wfn = optfreq(acetaldehyde, 'hf/sto-3g')

Optimizer: Optimization complete!


In [ ]:
# 振動数の出力
print(get_freqs(wfn))

[  162.82576484   545.47049956   896.29165205  1062.38919470
  1284.89067852  1293.96059028  1619.50704315  1703.94099902
  1806.46133073  1808.20706843  2121.29952716  3541.61964987
  3564.77447215  3739.48964642  3763.10227870]


In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('acetaldehyde_hf-sto3g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((162.8257648388, 0), (545.4704995565, 1), …

##### まとめて計算

In [ ]:
# 構造最適化と振動数計算の実行
co_aldehyde = []

for theory, logfile in zip(theories, logfiles):
    filename = f'acetaldehyde_{logfile}.log'
    psi4.set_output_file(filename)

    _, wfn = optfreq(acetaldehyde, theory)
    co_aldehyde.append(get_freqs(wfn)[10])

Optimizer: Optimization complete!
Optimizer: Optimization complete!
Optimizer: Optimization complete!
Optimizer: Optimization complete!


In [ ]:
print(co_aldehyde)
print(1727/np.array(co_aldehyde))

[np.float64(2121.4266268656434), np.float64(1926.1313141907895), np.float64(1648.36197157725), np.float64(1846.4468331893927)]
[ 0.81407482  0.89661592  1.04770677  0.93530990]


#### アセトアミド

In [ ]:
# アセトアミドの構造定義
acetamide = psi4.geometry('''
0 1
 C                  0.11203320    0.61825725    0.00000000
 O                  1.33489796    0.61545082   -0.10440743
 C                 -0.70916294    1.92103771    0.00000000
 H                 -0.40304404    2.53560774   -0.82066734
 H                 -1.74874842    1.68735489   -0.09774644
 H                 -0.54626743    2.44532874    0.91841383
 N                 -0.67183364   -0.62530452    0.00262330
 H                 -0.42379451   -1.17736195   -0.79343523
 H                 -0.47841040   -1.13949341    0.83820527
 ''')

##### HF/STO-3G

In [ ]:
psi4.set_output_file('acetamide_hf-sto3g.log')
_, wfn = optfreq(acetamide, 'hf/sto-3g')
print(get_freqs(wfn))

Optimizer: Optimization complete!
[  129.01293328   353.38391463   428.37635114   575.83437968
   582.65586634   817.53483120   960.59374183  1158.42164939
  1226.36058418  1350.98241403  1507.72263745  1703.36038108
  1804.69696406  1805.72946012  1950.88034297  2134.34971867
  3570.13079461  3746.41085363  3768.39806211  3952.13451607
  4153.16574527]


In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('acetamide_hf-sto3g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((129.0129332792, 0), (353.3839146273, 1), …

##### まとめて計算

In [ ]:
co_amide = []

for theory, logfile in zip(theories, logfiles):
    filename = f'acetamide_{logfile}.log'
    psi4.set_output_file(filename)

    _, wfn = optfreq(acetamide, theory)
    co_amide.append(get_freqs(wfn)[15])

Optimizer: Optimization complete!
Optimizer: Optimization complete!
Optimizer: Optimization complete!
Optimizer: Optimization complete!


In [ ]:
print(co_amide)
print(1681/np.array(co_amide))

[np.float64(2134.5473064827647), np.float64(1918.6556800491421), np.float64(1727.1009202081923), np.float64(1803.3708388381538)]
[ 0.78752061  0.87613427  0.97330734  0.93214328]


#### 塩化アセチル

In [ ]:
acetylchloride = psi4.geometry('''
0 1
 C                  0.11203320    0.61825725    0.00000000
 O                  1.33489796    0.61545082   -0.10440743
 C                 -0.70916294    1.92103771    0.00000000
 H                 -0.40304404    2.53560774   -0.82066734
 H                 -1.74874842    1.68735489   -0.09774644
 H                 -0.54626743    2.44532874    0.91841383
 Cl                -0.82647404   -0.87063304    0.00314082
''')

##### HF/STO-3G

In [ ]:
psi4.set_output_file('acetylchloride_hf-sto3g.log')
_, wfn = optfreq(acetylchloride, 'hf/sto-3g')
get_freqs(wfn)

Optimizer: Optimization complete!


array([  151.57586197,   349.36024613,   506.54231795,   544.99945674,
         728.54528011,  1140.75778320,  1227.10843611,  1306.96966823,
        1682.49204570,  1774.61867141,  1779.49469228,  2149.97963349,
        3573.56279104,  3755.19988847,  3765.54143396])

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('acetylchloride_hf-sto3g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((151.5758619658, 0), (349.3602461277, 1), …

##### まとめて計算

In [ ]:
co_chloride = []

for theory, logfile in zip(theories, logfiles):
    filename = f'acetylchloride_{logfile}.log'
    psi4.set_output_file(filename)

    _, wfn = optfreq(acetylchloride, theory)
    co_chloride.append(get_freqs(wfn)[11])

Optimizer: Optimization complete!
Optimizer: Optimization complete!
Optimizer: Optimization complete!
Optimizer: Optimization complete!


In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('acetylchloride_hf_3-21g.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((185.0432229528, 0), (347.5493681681, 1), …

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('acetylchloride_mp2_6-31g.C2ClH3O.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((142.6940291945, 0), (328.5441287326, 1), …

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('acetylchloride_wb97xd_aug-cc-pvdz.C2ClH3O.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((215.3555262502, 0), (359.8738361601, 1), …

In [ ]:
print(co_chloride)
1806/np.array(co_chloride)

[np.float64(2149.7644966757443), np.float64(2042.3444900715451), np.float64(1759.750996297698), np.float64(1917.8920159071615)]


array([ 0.84009202,  0.88427785,  1.02628156,  0.94165886])

### 熱力学的パラメータを求めてみよう

#### シクロプロパン

In [ ]:
psi4.set_options({'normal_modes_write': True})

In [ ]:
psi4.set_output_file('cyclopropane.log')

PosixPath('cyclopropane.log')

In [ ]:
cyclopropane = psi4.geometry('''
0 1
 C                  0.16445241    0.38365967    0.00286974
 C                  1.66556741    0.38365967    0.00286974
 C                  0.91498241    1.68363467    0.00286974
 H                 -0.37214359    0.07378467   -0.91098826
 H                 -0.37214359    0.07378467    0.91672774
 H                  2.20212841    0.07389267   -0.91101826
 H                  2.20212841    0.07389267    0.91675774
 H                  0.91475941    2.30316267    0.91681374
 H                  0.91475941    2.30316267   -0.91107426
 ''')

In [ ]:
energy_cpropane, wfn_cpropane = optfreq(cyclopropane, 'hf/cc-pvdz')

Optimizer: Optimization complete!


In [ ]:
print(f'cyclopropane freqs:\n{get_freqs(wfn_cpropane)}')
cyclopropane_H = psi4.variable('ENTHALPY')

cyclopropane freqs:
[  786.88029172   786.88241874   909.05249517   949.53336271
   949.54022733  1163.14660911  1163.15623084  1190.55496680
  1242.13186104  1292.04592124  1301.05053502  1301.05489002
  1566.04194025  1566.04310117  1638.16250754  3275.00843802
  3275.01061272  3291.17376035  3358.70267350  3358.70412017
  3381.62580985]


In [ ]:
show_3D(cyclopropane)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('cyclopropane.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((786.8802917169, 0), (786.8824187442, 1), …

#### メタン

In [ ]:
psi4.set_output_file('methane.log')

methane = psi4.geometry('''
0 1
 C                 -0.40248963    0.33609958    0.00000000
 H                 -0.04583521   -0.67271042    0.00000000
 H                 -0.04581679    0.84049777    0.87365150
 H                 -0.04581679    0.84049777   -0.87365150
 H                 -1.47248963    0.33611276    0.00000000
 ''')

In [ ]:
energy_methane, wfn_methane = optfreq(methane, 'hf/cc-pvdz')

Optimizer: Optimization complete!


In [ ]:
print(f'methane freqs:\n{get_freqs(wfn_methane)}')
methane_H = psi4.variable('ENTHALPY')

methane freqs:
[ 1433.99658095  1433.99663929  1433.99664965  1648.41291405
  1648.41292055  3164.08842297  3284.84479125  3284.84488645
  3284.84490214]


In [ ]:
show_3D(methane)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('methane.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((1433.996580953, 0), (1433.9966392875, 1),…

#### エタン

In [ ]:
psi4.set_output_file('ethane.log')

ethane = psi4.geometry('''
0 1
 C                 -0.40248963    0.33609958    0.00000000
 H                 -0.04583521   -0.67271042    0.00000000
 H                 -0.04581679    0.84049777   -0.87365150
 H                 -1.47248963    0.33611276    0.00000000
 C                  0.11085259    1.06205585    1.25740497
 H                  1.18085259    1.06202576    1.25741439
 H                 -0.24578624    2.07087137    1.25739542
 H                 -0.24583592    0.55766838    2.13105626
 ''')

In [ ]:
energy_ethane, wfn_ethane = optfreq(ethane, 'hf/cc-pvdz')

Optimizer: Optimization complete!


In [ ]:
print(f'ethane freqs:\n{get_freqs(wfn_ethane)}')
ethane_H = psi4.variable('ENTHALPY')

ethane freqs:
[  338.53274101   877.45009038   877.45009131  1062.08250824
  1313.01317088  1313.01317272  1501.90159256  1540.30886623
  1596.33333985  1596.33334125  1599.80893621  1599.80893758
  3169.45473881  3179.88530630  3230.34518398  3230.34519282
  3255.85038369  3255.85039275]


In [ ]:
show_3D(ethane)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('ethane.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((338.5327410052, 0), (877.4500903789, 1), …

#### アイソデスミック反応による推定

In [ ]:
au2kcal = psi4.constants.hartree2kcalmol

In [ ]:
isodesmic_h = au2kcal * (3 * ethane_H - (cyclopropane_H + 3 * methane_H))
print(f'estimated strain energy using isodesmic reaction: {-isodesmic_h: .2f} kcal/mol')

estimated strain energy using isodesmic reaction:  21.88 kcal/mol


#### メチルシクロヘキサン

エクアトリアル置換

In [ ]:
psi4.set_output_file('equatorial.log')

PosixPath('equatorial.log')

In [ ]:
equatorial = psi4.geometry('''
0 1
 C                 -2.15838901   -1.67295376    0.00000000
 C                 -0.64328301   -1.67295376    0.00000000
 C                 -0.09135201   -0.26187576    0.00000000
 C                 -0.64101501    0.54266124    1.16066100
 C                 -2.15614001    0.54332224    1.16017200
 C                 -2.70894001   -0.86729876    1.15887600
 H                  1.02724699   -0.29586676    0.06271400
 H                 -0.27073401   -2.21858776    0.90656200
 H                 -0.26798901   -2.22281276   -0.90191000
 H                 -2.53107701   -1.23992476   -0.96538500
 H                 -2.53398601   -2.72717376    0.06350200
 H                 -0.26848001    0.10781624    2.12527200
 H                 -0.26499201    1.59679824    1.09866600
 H                 -2.53146001    1.09275024    2.06228600
 H                 -2.52799501    1.08992324    0.25384900
 H                 -2.44293601   -1.37302076    2.12419100
 H                 -0.35997001    0.24375724   -0.96454600
 C                 -4.24559913   -0.81913202    1.06966430
 H                 -4.63527674   -1.81550142    1.08696717
 H                 -4.63411695   -0.26971208    1.90158475
 H                 -4.53508198   -0.33871606    0.15845621
 ''')

In [ ]:
_, equatorial_wfn = optfreq(equatorial, 'hf/cc-pvdz')

Optimizer: Optimization complete!
 '468.9494' '478.7244' '580.0607']


In [ ]:
print(f'equatorial freqs:\n{get_freqs(equatorial_wfn)}')
equatorial_G = psi4.variable('GIBBS FREE ENERGY')
equatorial_Gcorr = psi4.variable('GIBBS FREE ENERGY CORRECTION')

equatorial freqs:
[  162.81963630   239.28007629   258.83751507   330.19198010
   351.41685642   434.56650627   468.94938046   478.72442995
   580.06074814   823.90444277   851.91709879   903.27639243
   923.65444575   938.10620065   990.15403029  1043.10708828
  1050.88690087  1051.53735597  1111.65167182  1145.71450452
  1165.58273234  1189.75594296  1221.55833882  1283.53584436
  1321.54284250  1372.09724912  1384.15280778  1389.20309115
  1437.52937579  1440.67697685  1476.92144637  1502.21571163
  1502.79364085  1508.86335919  1518.81999722  1532.72026072
  1581.03570932  1588.14909111  1593.64208186  1594.06630092
  1595.32025829  1597.07552731  1613.26198880  3131.74033559
  3145.60618738  3151.55454920  3154.64783403  3155.09928783
  3157.30174536  3165.39802994  3197.20768796  3199.95011187
  3204.24610082  3207.14186692  3213.30180608  3234.38844998
  3237.78763726]


In [ ]:
print(equatorial.save_string_xyz())

0 1
 C   -0.336905645260   -1.071358280084   -0.668503623813
 C    1.193285528804   -1.083213559991   -0.655705550409
 C    1.767015747030    0.334784442141   -0.641628681364
 C    1.194920051650    1.154460177893    0.516354217332
 C   -0.335275998257    1.159982347078    0.500243315967
 C   -0.921697826422   -0.257677573362    0.493237543116
 H    2.858119395475    0.300824427809   -0.578316212939
 H    1.542402302476   -1.622171770084    0.232717418798
 H    1.571078203300   -1.635298796078   -1.520599758231
 H   -0.687174009651   -0.642514210202   -1.615726575620
 H   -0.718920716604   -2.095958807853   -0.633566109877
 H    1.544124036416    0.730638046878    1.465082215414
 H    1.573861886160    2.179378813222    0.477467785965
 H   -0.716135747103    1.715129126321    1.362635389061
 H   -0.685469607696    1.694851518308   -0.391444105526
 H   -0.614483494431   -0.747006887715    1.427020851369
 H    1.525589786423    0.831146705010   -1.588932130148
 C   -2.448953060642   -0.2

In [ ]:
show_3D(equatorial)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('equatorial.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((162.8196363034, 0), (239.2800762868, 1), …

アキシアル置換

In [ ]:
psi4.set_output_file('axial.log')

PosixPath('axial.log')

In [ ]:
axial = psi4.geometry('''
0 1
 C                 -2.15838901   -1.67295376    0.00000000
 C                 -0.64328301   -1.67295376    0.00000000
 C                 -0.09135201   -0.26187576    0.00000000
 C                 -0.64101501    0.54266124    1.16066100
 C                 -2.15614001    0.54332224    1.16017200
 C                 -2.70894001   -0.86729876    1.15887600
 H                  1.02724699   -0.29586676    0.06271400
 H                 -0.27073401   -2.21858776    0.90656200
 H                 -0.26798901   -2.22281276   -0.90191000
 H                 -2.53107701   -1.23992476   -0.96538500
 H                 -2.53398601   -2.72717376    0.06350200
 H                 -0.26848001    0.10781624    2.12527200
 H                 -0.26499201    1.59679824    1.09866600
 H                 -2.53146001    1.09275024    2.06228600
 H                 -2.52799501    1.08992324    0.25384900
 H                 -0.35997001    0.24375724   -0.96454600
 C                 -2.34375843   -1.56157538    2.48410141
 H                 -2.74805066   -1.00326314    3.30247494
 H                 -2.75034420   -2.55128438    2.49210909
 H                 -1.27915043   -1.61256543    2.57849329
 H                 -3.77661875   -0.83383226    1.09689125
 ''')

In [ ]:
_, axial_wfn = optfreq(axial, 'hf/cc-pvdz')

Optimizer: Optimization complete!
 '481.8137' '510.3954']


In [ ]:
print(f'axial freqs:\n{get_freqs(equatorial_wfn)}')
axial_G = psi4.variable('GIBBS FREE ENERGY')
axial_Gcorr = psi4.variable('GIBBS FREE ENERGY CORRECTION')

axial freqs:
[  162.81963630   239.28007629   258.83751507   330.19198010
   351.41685642   434.56650627   468.94938046   478.72442995
   580.06074814   823.90444277   851.91709879   903.27639243
   923.65444575   938.10620065   990.15403029  1043.10708828
  1050.88690087  1051.53735597  1111.65167182  1145.71450452
  1165.58273234  1189.75594296  1221.55833882  1283.53584436
  1321.54284250  1372.09724912  1384.15280778  1389.20309115
  1437.52937579  1440.67697685  1476.92144637  1502.21571163
  1502.79364085  1508.86335919  1518.81999722  1532.72026072
  1581.03570932  1588.14909111  1593.64208186  1594.06630092
  1595.32025829  1597.07552731  1613.26198880  3131.74033559
  3145.60618738  3151.55454920  3154.64783403  3155.09928783
  3157.30174536  3165.39802994  3197.20768796  3199.95011187
  3204.24610082  3207.14186692  3213.30180608  3234.38844998
  3237.78763726]


In [ ]:
print(axial.save_string_xyz())

0 1
 C   -0.599543286121   -0.980802814131   -0.843568635273
 C    0.930382140781   -0.947542055212   -0.917636367715
 C    1.457874852662    0.488881366616   -0.935599767485
 C    0.931966910293    1.292921006395    0.255544853317
 C   -0.597963767940    1.252686352128    0.325960889508
 C   -1.167375354888   -0.177045623495    0.339688019471
 H    2.551390571189    0.490941063615   -0.941013779496
 H    1.359362064606   -1.481122508548   -0.063182859760
 H    1.266671834250   -1.482607527662   -1.810079066165
 H   -1.001474795870   -0.562054272522   -1.773109785420
 H   -0.952230681371   -2.015342497996   -0.795565274895
 H    1.361039520961    0.894100211221    1.180564189762
 H    1.269369744018    2.330836904418    0.186769958909
 H   -0.949534238775    1.802923117833    1.203801342648
 H   -0.999814789989    1.778597984884   -0.547467932052
 H    1.141500811798    0.976109684211   -1.865640954443
 C   -0.959583863050   -0.881795033691    1.685282042594
 H   -1.419766526040   -0.3

In [ ]:
show_3D(axial)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 振動数を可視化（左サイドバーから.molden_normal_modesファイルの名前を参照し指定する。ファイル名の番号は実行時のOSプロセスID）
show_normal_modes('axial.default.1611.molden_normal_modes')

interactive(children=(Dropdown(description='Normal mode:', options=((170.3747785459, 0), (216.4341678219, 1), …

##### 自由エネルギー差を推定

In [ ]:
au2kcal = psi4.constants.hartree2kcalmol
diff = au2kcal * (axial_G - equatorial_G)
print(f'free energy difference: {diff: .2f} kcal/mol')

free energy difference:  2.68 kcal/mol


### 高精度でエネルギーを求める場合

In [ ]:
print(equatorial_Gcorr, axial_Gcorr)

0.17953219821029034 0.1799284596223497


In [ ]:
high_level = 'mp2/aug-cc-pvtz'

psi4.set_output_file('high_level_single_point.log')

eq_energy = psi4.energy(high_level, molecule=equatorial)
ax_energy = psi4.energy(high_level, molecule=axial)

In [ ]:
eq_free_energy = eq_energy + equatorial_Gcorr
ax_free_energy = ax_energy + axial_Gcorr

In [ ]:
diff_free_energy = (ax_free_energy - eq_free_energy) * au2kcal
print(f'free energy difference at high level: {diff_free_energy: .2f} kcal/mol')

free energy difference at high level:  2.06 kcal/mol
